In [8]:
import os

path = r"D:\mdr-pepdesign\docking\results\results"
print("Contents:", os.listdir(path)[:10])
print("Total items:", len(os.listdir(path)))

Contents: ['pep_0001_log.txt', 'pep_0001_out.pdbqt', 'pep_0002_log.txt', 'pep_0002_out.pdbqt', 'pep_0004_log.txt', 'pep_0004_out.pdbqt', 'pep_0005_log.txt', 'pep_0005_out.pdbqt', 'pep_0006_log.txt', 'pep_0006_out.pdbqt']
Total items: 578


In [9]:
import shutil, os

src = r"D:\mdr-pepdesign\docking\results\results"
dst = r"D:\mdr-pepdesign\docking\results"

for f in os.listdir(src):
    shutil.move(os.path.join(src, f), os.path.join(dst, f))

os.rmdir(src)
print("Moved files up one level, removed empty nested folder")
print("Now in docking/results:", len(os.listdir(dst)))

Moved files up one level, removed empty nested folder
Now in docking/results: 579


In [10]:
"""
Generate combined receptor+peptide complex PDB files for the 3D viewer.
============================================================================
viewer.py expects: data/complex_top20/{pep_id}.pdb
  - One PDB file per top candidate
  - Receptor on one chain, peptide on a different chain (default "C")
  - This is what makes the dashboard's "swap between inhibitors" feature work

INPUT (already exists from today's docking run):
  - docking/receptor_af.pdbqt  (the AlphaFold-based EmrE receptor)
  - docking/results/{pep_id}_out.pdbqt  (Vina's docked pose, already in the
    receptor's coordinate frame - no alignment needed, just concatenation)

WHAT THIS DOES:
  1. Reads the receptor, strips PDBQT-specific columns down to plain PDB format,
     assigns it chain A (or whatever the protein already uses)
  2. Reads each top candidate's docked pose (mode 1 / best pose only - Vina's
     output PDBQT contains multiple MODEL blocks, one per binding mode)
  3. Relabels the peptide's chain to "C" (matching PEPTIDE_CHAIN in utils/data.py)
  4. Concatenates into a single valid PDB file
  5. Saves to data/complex_top20/{pep_id}.pdb

Run this ONCE after docking completes, or whenever you want to refresh the
set of "top 20" complexes shown in the dashboard.
"""

import os
import pandas as pd

# ============================================================
# SETTINGS
# ============================================================
PROJECT_ROOT   = r"D:\mdr-pepdesign"   # change for Colab: leave as relative paths instead
RECEPTOR_PATH  = "docking/receptor_af.pdbqt"
RESULTS_DIR    = "docking/results"
SCORES_CSV     = "data/docking_scores.csv"
OUTPUT_DIR     = "data/complex_top20"
TOP_N          = 20
RECEPTOR_CHAIN = "A"
PEPTIDE_CHAIN  = "C"   # must match PEPTIDE_CHAIN in utils/data.py
# ============================================================


def pdbqt_atom_to_pdb_line(line, chain_override=None):
    """
    Convert a single PDBQT ATOM/HETATM line to a clean PDB ATOM line.
    PDBQT format is PDB format + two extra columns (charge, AD atom type)
    appended after the standard PDB columns - we just truncate those off
    and optionally relabel the chain.
    """
    if not (line.startswith("ATOM") or line.startswith("HETATM")):
        return None

    # Standard PDB columns are the first 66 characters (through B-factor);
    # PDBQT appends charge + atom type after that - we drop those extra columns.
    pdb_line = line[:66]

    # Pad to standard 80-char PDB line length, replacing PDBQT's extra columns
    pdb_line = pdb_line.ljust(66)

    if chain_override:
        # Chain ID is column 22 (0-indexed 21) in standard PDB format
        pdb_line = pdb_line[:21] + chain_override + pdb_line[22:]

    # Always write as ATOM (not HETATM) so viewers render it as part of the
    # polymer chain rather than as a disconnected ligand/heteroatom group
    pdb_line = "ATOM  " + pdb_line[6:]

    return pdb_line.rstrip() + "\n"


def extract_best_pose(pdbqt_path):
    """
    Vina's output PDBQT contains multiple MODEL/ENDMDL blocks, one per
    binding mode, ranked best-to-worst. We only want mode 1 (the best pose)
    for the viewer - extract just the first MODEL block.
    """
    with open(pdbqt_path) as f:
        lines = f.readlines()

    in_first_model = False
    model_lines = []
    seen_first_model = False

    for line in lines:
        if line.startswith("MODEL"):
            if seen_first_model:
                break  # stop at the start of the second model
            in_first_model = True
            seen_first_model = True
            continue
        if line.startswith("ENDMDL"):
            break
        if in_first_model:
            model_lines.append(line)

    return model_lines


def build_receptor_pdb_lines():
    """Convert the receptor PDBQT to clean PDB-format lines once, reused
    for every complex file (the receptor doesn't change between peptides)."""
    with open(RECEPTOR_PATH) as f:
        lines = f.readlines()

    pdb_lines = []
    for line in lines:
        converted = pdbqt_atom_to_pdb_line(line, chain_override=RECEPTOR_CHAIN)
        if converted:
            pdb_lines.append(converted)

    pdb_lines.append("TER\n")
    return pdb_lines


def build_complex(pep_id, receptor_lines):
    """Combine the cached receptor lines with one peptide's best docked pose."""
    pose_path = os.path.join(RESULTS_DIR, f"{pep_id}_out.pdbqt")
    if not os.path.exists(pose_path):
        return None, "pose file missing"

    pose_lines = extract_best_pose(pose_path)
    if not pose_lines:
        return None, "no MODEL block found in pose file"

    peptide_pdb_lines = []
    for line in pose_lines:
        converted = pdbqt_atom_to_pdb_line(line, chain_override=PEPTIDE_CHAIN)
        if converted:
            peptide_pdb_lines.append(converted)

    if not peptide_pdb_lines:
        return None, "no atoms extracted from pose"

    complex_lines = receptor_lines + peptide_pdb_lines + ["TER\n", "END\n"]
    return complex_lines, "ok"


def main():
    os.chdir(PROJECT_ROOT)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if not os.path.exists(RECEPTOR_PATH):
        print(f"ERROR: receptor not found at {RECEPTOR_PATH}")
        return

    if not os.path.exists(SCORES_CSV):
        print(f"ERROR: {SCORES_CSV} not found - run parse_results.py first")
        return

    scores = pd.read_csv(SCORES_CSV).sort_values("dG").reset_index(drop=True)
    top_ids = scores.head(TOP_N)["id"].tolist()

    print(f"Building complex files for top {len(top_ids)} candidates...")
    receptor_lines = build_receptor_pdb_lines()
    print(f"Receptor converted: {len(receptor_lines)} lines")

    built, failed = 0, []
    for pep_id in top_ids:
        complex_lines, status = build_complex(pep_id, receptor_lines)
        if complex_lines is None:
            failed.append((pep_id, status))
            print(f"  FAIL  {pep_id}: {status}")
            continue

        out_path = os.path.join(OUTPUT_DIR, f"{pep_id}.pdb")
        with open(out_path, "w") as f:
            f.writelines(complex_lines)
        built += 1

    print(f"\n{'='*40}")
    print(f"Complex files built: {built}/{len(top_ids)}")
    print(f"Saved to: {OUTPUT_DIR}/")
    if failed:
        print(f"Failed: {len(failed)}")
        for pep_id, reason in failed:
            print(f"  {pep_id}: {reason}")


if __name__ == "__main__":
    main()

Building complex files for top 20 candidates...
Receptor converted: 1000 lines

Complex files built: 20/20
Saved to: data/complex_top20/


In [11]:
st.write(complex_path)
st.write(os.path.exists(complex_path))

NameError: name 'st' is not defined